# Phase 2 — LoRA Fine-Tuning of NLLB-200 for Catanduanes (Virac) Bikol → English

**Capstone:** Low-Resource NMT for Bikol Dialect Variants · **Author:** Lenard Benavidez (BSCS-3B, CatSU)

This notebook fine-tunes `facebook/nllb-200-distilled-600M` on the Phase 1 corpus using **LoRA**,
produces the **training/validation loss curve** (Figure 1, required by the rubric), and saves the
**LoRA adapter** plus a reload test.

### Before you run
1. In Colab: **Runtime → Change runtime type → T4 GPU**, then Save.
2. Run the cells top to bottom.
3. The **test set is NOT used here** — it is touched only once, in Phase 4 evaluation. This notebook
   trains on `train` and tunes/monitors on `val` only (rubric criterion B5: method safeguards).

> Note on language codes: NLLB has no code for Catanduanes Bikol. We reuse the closest supported
> code, **`bcl_Latn` (Central Bikol)**, as the source tag and fine-tune the model to adapt it to the
> Virac variant. This is the standard low-resource transfer approach and is exactly why the project
> exists (NLLB covers `bcl` but not the Catanduanes variants).

## 1. Install dependencies

In [ ]:
# Pinned, Colab-friendly versions. ~2-3 min on first run.
!pip -q install "transformers==4.44.2" "peft==0.12.0" "datasets==2.21.0" \
    "sacrebleu==2.4.3" "accelerate==0.33.0" sentencepiece matplotlib
print("done")

## 2. Imports, reproducibility seed, and configuration
Every hyperparameter below comes from the project brief. The seed makes the run reproducible (rubric B5).

In [ ]:
import os, random, json
import numpy as np
import torch
import matplotlib.pyplot as plt

# ---- Reproducibility (B5: seed recorded in code) ----
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---- Model & language codes ----
BASE_MODEL = "facebook/nllb-200-distilled-600M"  # B1: 600M fits a free T4; already covers Central Bikol
SRC_LANG   = "bcl_Latn"   # closest supported code, reused for the Virac variant
TGT_LANG   = "eng_Latn"
MAX_LEN    = 128

# ---- LoRA settings (B2 — straight from the brief) ----
LORA_R        = 16          # brief: 8-16  -> 16 for a bit more capacity on a small corpus
LORA_ALPHA    = 32          # brief: 16-32
LORA_DROPOUT  = 0.05        # brief: 0.05
TARGET_MODULES = ["q_proj", "v_proj"]   # brief: attention query & value projections

# ---- Training settings (B3 — from the brief) ----
LEARNING_RATE = 3e-4        # brief: 3e-4
NUM_EPOCHS    = 20          # brief: 15-30 (watch the val-loss curve; stop if it rises)
BATCH_SIZE    = 16
WARMUP_RATIO  = 0.06
WEIGHT_DECAY  = 0.01

OUT_DIR    = "nllb_bikol_lora"       # where the adapter is saved
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
assert device == "cuda", "No GPU! Set Runtime -> Change runtime type -> T4 GPU."

## 3. Load the corpus from your GitHub repo
Pulls the exact Phase 1 split you pushed, so the notebook is reproducible from the repo alone.

In [ ]:
!rm -rf bikol-nmt && git clone -q https://github.com/LenardBenavidez/bikol-nmt.git
import pandas as pd

PS = "bikol-nmt/data/processed_sentences"
train_df = pd.read_csv(f"{PS}/train.csv")
val_df   = pd.read_csv(f"{PS}/val.csv")
test_df  = pd.read_csv(f"{PS}/test.csv")   # loaded for reference only; NOT used in training

SRC_COL, TGT_COL = "Bicol", "English"      # column names in the corpus
for df in (train_df, val_df, test_df):
    df.dropna(subset=[SRC_COL, TGT_COL], inplace=True)

print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)} (test held out)")
train_df.head()

## 4. Load NLLB-200 and the tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

# Force English as the first generated token at inference time.
model.config.forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
print("loaded:", BASE_MODEL)

## 5. Apply LoRA
We freeze the 600M base weights and train only small LoRA adapters on the attention q/v projections. This is what keeps training cheap enough for a free GPU.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # expect <1% of params trainable

## 6. Tokenize the data

In [ ]:
from datasets import Dataset

def preprocess(batch):
    # src_lang/tgt_lang are already set on the tokenizer, so the right language
    # tokens are prepended automatically.
    model_inputs = tokenizer(batch[SRC_COL], max_length=MAX_LEN, truncation=True)
    labels = tokenizer(text_target=batch[TGT_COL], max_length=MAX_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = Dataset.from_pandas(train_df[[SRC_COL, TGT_COL]]).map(
    preprocess, batched=True, remove_columns=[SRC_COL, TGT_COL])
val_ds = Dataset.from_pandas(val_df[[SRC_COL, TGT_COL]]).map(
    preprocess, batched=True, remove_columns=[SRC_COL, TGT_COL])
print(train_ds, val_ds)

## 7. Train
Validation loss is computed every epoch so we can plot the curve and spot overfitting (B3, B5). Hyperparameters are tuned on the **validation** set only — the test set is never seen here.

In [ ]:
from transformers import (Seq2SeqTrainingArguments, Seq2SeqTrainer,
                          DataCollatorForSeq2Seq, EarlyStoppingCallback)

collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    optim="adamw_torch",              # brief: AdamW
    lr_scheduler_type="linear",       # brief: linear decay
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,      # keep the epoch with the lowest val loss
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    seed=SEED,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator, tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
trainer.train()

## 8. Figure 1 — Training & validation loss per epoch *(required)*

In [ ]:
hist = trainer.state.log_history
tr = {e["epoch"]: e["loss"]      for e in hist if "loss" in e}
va = {e["epoch"]: e["eval_loss"] for e in hist if "eval_loss" in e}

plt.figure(figsize=(7,4.5))
plt.plot(list(tr.keys()), list(tr.values()), marker="o", label="Training loss")
plt.plot(list(va.keys()), list(va.values()), marker="s", label="Validation loss")
best_ep = min(va, key=va.get)
plt.axvline(best_ep, color="gray", ls="--", lw=1)
plt.text(best_ep, max(list(tr.values())+list(va.values())), f" best epoch {best_ep:.0f}",
         va="top", fontsize=9, color="gray")
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy loss")
plt.title("Figure 1. Training and validation loss per epoch")
plt.legend(); plt.tight_layout()
plt.savefig("training_loss_curve.png", dpi=150)
plt.show()
print("saved training_loss_curve.png  | best (lowest val-loss) epoch:", best_ep)

## 9. Save the LoRA adapter (B4)

In [ ]:
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

# report adapter size
sz = 0
for root,_,files in os.walk(OUT_DIR):
    for f in files:
        if f.endswith((".safetensors",".bin")): sz += os.path.getsize(os.path.join(root,f))
print(f"adapter saved to '{OUT_DIR}/'  (~{sz/1e6:.1f} MB)")
!ls -la {OUT_DIR}

## 10. Reload test (B4)
Load the base model + your saved adapter from scratch and translate a sample sentence — proof the adapter works independently.

In [ ]:
from peft import PeftModel

base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
reloaded = PeftModel.from_pretrained(base, OUT_DIR).to(device).eval()
tok = AutoTokenizer.from_pretrained(OUT_DIR, src_lang=SRC_LANG, tgt_lang=TGT_LANG)

def translate(text):
    enc = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(device)
    out = reloaded.generate(**enc,
            forced_bos_token_id=tok.convert_tokens_to_ids(TGT_LANG), max_length=MAX_LEN)
    return tok.batch_decode(out, skip_special_tokens=True)[0]

sample = val_df.iloc[0]
print("Bikol in :", sample[SRC_COL])
print("Reference:", sample[TGT_COL])
print("Model out:", translate(sample[SRC_COL]))

## 11. Download your results
Saves the adapter + loss curve as a zip. Download it, unzip into your repo, and let Claude Code commit it (e.g. under `results/` and `models/`).

In [ ]:
!zip -q -r phase2_outputs.zip {OUT_DIR} training_loss_curve.png
from google.colab import files
files.download("phase2_outputs.zip")

---
### What to record for the report (Section 3, criteria B1–B5)
- **B1 base model:** `facebook/nllb-200-distilled-600M` — fits a free T4; already supports Central Bikol (`bcl`) as a transfer base.
- **B2 LoRA:** r=16, α=32, dropout=0.05, target modules q_proj/v_proj.
- **B3 training:** lr=3e-4, AdamW, linear decay, warmup 6%, up to 20 epochs — see Figure 1; note the best (lowest val-loss) epoch.
- **B4 export/reload:** adapter size from cell 9; reload-test output from cell 10.
- **B5 safeguards:** hyperparameters chosen on the **validation** set; the **test** set is untouched until Phase 4; seed = 42, set in cell 2.